# Streaming & Performance Cheatsheet

Quick reference for Spark Structured Streaming and performance tuning.

## Table of Contents
- [Structured Streaming Basics](#structured-streaming-basics)
- [Streaming Operations](#streaming-operations)
- [Window Operations](#window-operations)
- [Performance Tuning](#performance-tuning)
- [Monitoring & Debugging](#monitoring--debugging)

## Structured Streaming Basics

In [ ]:
# Create streaming DataFrame
streaming_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "topic") \
    .load()

# Read from files
streaming_df = spark.readStream \
    .schema(schema) \
    .json("path/to/files")

# Read from socket
streaming_df = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# Write stream to console
query = streaming_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

# Write stream to file
query = streaming_df.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "path/to/output") \
    .option("checkpointLocation", "path/to/checkpoint") \
    .start()

# Write stream to memory
query = streaming_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("table_name") \
    .start()

## Output Modes

In [ ]:
# Append mode (default)
# Only new rows are output
query = streaming_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

# Complete mode
# All rows are output (full state)
query = streaming_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

# Update mode
# Only updated rows are output
query = streaming_df.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

## Streaming Operations

In [ ]:
from pyspark.sql.functions import col, explode, split

# Parse JSON data
parsed_df = streaming_df.select(
    explode(split(col("value"), ",")).alias("word")
)

# Filter streaming data
filtered_df = streaming_df.filter(col("value").isNotNull())

# Aggregations on streaming data
from pyspark.sql.functions import count, window

word_counts = parsed_df \
    .groupBy("word") \
    .count() \
    .orderBy(col("count").desc())

## Window Operations

In [ ]:
from pyspark.sql.functions import window, col

# Tumbling window
windowed_counts = parsed_df \
    .groupBy(window(col("timestamp"), "1 minute")) \
    .count()

# Sliding window
windowed_counts = parsed_df \
    .groupBy(window(col("timestamp"), "1 minute", "30 seconds")) \
    .count()

# Session window
windowed_counts = parsed_df \
    .groupBy(session_window(col("timestamp"), "5 minutes")) \
    .count()

## Watermarking

In [ ]:
# Add watermark to handle late data
from pyspark.sql.functions import window

watermarked_df = streaming_df.withWatermark(
    "timestamp", "10 minutes"
)

# Window with watermark
windowed_counts = watermarked_df \
    .groupBy(
        window(col("timestamp"), "5 minutes"),
        "word"
    ) \
    .count()

## Query Management

In [ ]:
# Start query
query = streaming_df.writeStream.start()

# Wait for query to finish
query.awaitTermination()

# Wait with timeout
query.awaitTermination(timeout=60)

# Stop query
query.stop()

# Query status
query.status
query.lastProgress
query.recentProgress

# List active queries
spark.streams.active

# Get query by name
query = spark.streams.get("query_name")

## Performance Tuning

In [ ]:
# Shuffle partitions for streaming
spark.conf.set("spark.sql.shuffle.partitions", "10")

# Enable adaptive query execution
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Enable continuous processing (experimental)
query = streaming_df.writeStream \
    .format("console") \
    .option("trigger", "continuous") \
    .start()

# Trigger interval for micro-batch
query = streaming_df.writeStream \
    .format("console") \
    .trigger(processingTime="1 minute") \
    .start()

# Once trigger (useful for testing)
query = streaming_df.writeStream \
    .format("console") \
    .trigger(once=True) \
    .start()

## Performance Tuning - General

In [ ]:
# Memory configuration
spark.conf.set("spark.executor.memory", "4g")
spark.conf.set("spark.driver.memory", "2g")
spark.conf.set("spark.memory.fraction", "0.6")
spark.conf.set("spark.memory.storageFraction", "0.5")

# Parallelism
spark.conf.set("spark.default.parallelism", "8")
spark.conf.set("spark.sql.shuffle.partitions", "200")

# Serialization
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

# Compression
spark.conf.set("spark.io.compression.codec", "snappy")
spark.conf.set("spark.rdd.compress", "true")

# Broadcast join threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10MB

# Adaptive query execution
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

## Monitoring & Debugging

In [ ]:
# Explain query plan
streaming_df.explain()
streaming_df.explain(extended=True)

# Query execution metrics
query.lastProgress
query.recentProgress

# Spark UI
# Access at http://localhost:4040
# Check: Jobs, Stages, Storage, SQL, Streaming tabs

# Logging configuration
spark.sparkContext.setLogLevel("INFO")
spark.sparkContext.setLogLevel("DEBUG")

# Accumulators for custom metrics
accum = spark.sparkContext.accumulator(0)

def increment_accum(x):
    accum.add(1)
    return x

# Use in map/foreach
rdd.foreach(increment_accum)
print(f"Processed {accum.value} records")

## Common Performance Issues

### Data Skew
- **Symptom**: One task takes much longer than others
- **Solution**: Add salt keys, broadcast small tables, increase partitions

### Small Files Problem
- **Symptom**: Many small files causing overhead
- **Solution**: Use repartition/coalesce before writing, use compaction

### Memory Issues
- **Symptom**: Out of memory errors
- **Solution**: Increase executor memory, use more partitions, cache selectively

### Slow Joins
- **Symptom**: Join operations taking too long
- **Solution**: Use broadcast joins for small tables, increase shuffle partitions